In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plot

%matplotlib inline

In [ ]:
def f(x):
    return 3 * x**2 - 4 * x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plot.plot(xs, ys)
plot.show()

In [ ]:
h = 0.000001
e = 3
(f(e + h) - f(e)) / h

In [ ]:
h = 0.000001
e = -3
(f(e + h) - f(e)) / h

In [ ]:
h = 0.00001
e = 2 / 3
(f(e + h) - f(e)) / h

In [ ]:
a = 2.0
b = -3.0
c = 10.0

h = 0.001
d1 = a * b + c
print(d1)
d2 = (a + h) * b + c

# Rate of change of a => dd/da = b
(d2 - d1) / h

In [ ]:
class Value:
    def __init__(self, data, _children=(), op="", label="") -> None:
        self.data = data
        self._prev = set(_children)  # What all objects are involved to form this Object
        self._op = op  # What operation was performed to get this data
        self.label = label
        self.grad: float = 0
        self._backward = lambda: None

    def __repr__(self) -> str:
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = self.__type_validate_and_convert(other)
        out = Value(self.data + other.data, (self, other), "+")

        def backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = backward

        return out

    def __sub__(self, other):
        return self + (-other)

    def __neg__(self):
        return self * -1

    def __mul__(self, other):
        other = self.__type_validate_and_convert(other)
        out = Value(self.data * other.data, (self, other), "*")

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = self.__type_validate_and_convert(other)
        return self * (other**-1)

    def __rtruediv__(self, other):
        other = self.__type_validate_and_convert(other)
        return other / self

    def __pow__(self, other):
        assert isinstance(other, (int, float)), (
            "only supported int/float powers for now"
        )
        out = Value(
            self.data**other,
            (self,),
            f"**{other}",
        )

        def backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), "tanh")

        def backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = backward
        return out

    def exp(self):
        x = self.data
        t = math.exp(x)
        out = Value(t, (self,), "exp")

        def backward():
            self.grad += t * out.grad

        out._backward = backward
        return out

    def backward(self):
        toposort = []
        visited = set()

        def build_topological_sort(node: Value):
            if node in visited:
                return

            visited.add(node)
            for ch in node._prev:
                build_topological_sort(ch)

            toposort.append(node)

        build_topological_sort(self)

        for node in toposort:
            node.grad = 0.0

        self.grad = 1.0

        for node in reversed(toposort):
            node._backward()

    @staticmethod
    def __type_validate_and_convert(other):
        return other if isinstance(other, Value) else Value(other)


In [ ]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in sorted(v._prev, key=lambda node: id(node)):
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})
    nodes, edges = trace(root)

    for n in sorted(nodes, key=lambda node: id(node)):
        uid = str(id(n))

        dot.node(
            name=uid,
            label=f"{{ {n.label} | data: {n.data:.4f} | grad: {n.grad:.4f} }}",
            shape="record",
        )
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in sorted(edges, key=lambda edge: (id(edge[0]), id(edge[1]))):
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
a = Value(2)
b = Value(4)
a / b

# Back Propagation examples


## Example 1


In [ ]:
a = Value(2, label="a")
b = Value(-3, label="b")

c = Value(10, label="c")
e = a * b
e.label = "e"

d = e + c
d.label = "d"

f = Value(-2, label="f")

L = d * f
L.label = "L"

L.backward()
draw_dot(L)

## tanh graph


In [ ]:
plot.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2)))
plot.grid()


## Example 2


In [ ]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.88137, label="b")

x1w1 = x1 * w1
x1w1.label = "x1w1"
x2w2 = x2 * w2
x2w2.label = "x2w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = "x1w1x2w2"

n = x1w1x2w2 + b
n.label = "x1w1x2w2 + b"

o = n.tanh()
o.label = "o"

o.backward()
draw_dot(o)

In [ ]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.88137, label="b")

x1w1 = x1 * w1
x1w1.label = "x1w1"
x2w2 = x2 * w2
x2w2.label = "x2w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = "x1w1x2w2"

n = x1w1x2w2 + b
n.label = "x1w1x2w2 + b"

# ===
e = (2 * n).exp()
o = (e - 1) / (e + 1)
# ===

o.label = "o"
o.backward()
draw_dot(o)


### Manual Back propagation of example 2


In [ ]:
n.grad = 1 - o.data**2  ## d(tanh(x))/dx = 1 - tanh(x)^2

x1w1x2w2.grad = n.grad
b.grad = n.grad

x1w1.grad = x1w1x2w2.grad
x2w2.grad = x1w1x2w2.grad

x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad

x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

# Evaluate same expression using pytorch


In [ ]:
import torch

In [ ]:
x1 = torch.tensor([2.0]).double()
x2 = torch.tensor([0.0]).double()
w1 = torch.tensor([-3.0]).double()
w2 = torch.tensor([1.0]).double()
b = torch.tensor([6.88137]).double()
x1.requires_grad = True
x2.requires_grad = True
w1.requires_grad = True
w2.requires_grad = True
b.requires_grad = True

n = x1 * w1 + x2 * w2 + b
o = n.tanh()


o.backward()

print(o.data.item(), end="\n-------\n")
print(x1.grad.item())
print(x2.grad.item())
print(w1.grad.item())
print(x2.grad.item())
